Processar o arquivo de base com as perguntas.

In [ ]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [ ]:
# Instalar ou atualizar as bibliotecas necessárias.
!pip install -q -U openai bert-score

# Importar bibliotecas.
import pandas as pd
import os
from openai import OpenAI
from google import genai
from google.colab import userdata
from bert_score import score

Funcões dos clientes das IAs, modelos e realização de consultas.

In [ ]:
# Iniciar o cliente da IA
def client_ai_instance(name_api_key):
  # Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
  # O uso do Secrets é uma alternativa para que chave de API não fique exposta no código.
  # Tal chave previamente criada no Google AI Studio ou no Console do Groq.
  # Observação: o nome da chave definido precisa ser o mesmo inclusive com diferenciação de letra maiúscula e minúscula.
  api_key = userdata.get(name_api_key)

  # Verificar se foi o groq ou google.
  if name_api_key == 'groq_api_key':
    client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=api_key
  )
  elif name_api_key == 'oprouter_api_key':
    client_ai = OpenAI(
      base_url="https://openrouter.ai/api/v1/",
      api_key=api_key
    )
  elif name_api_key == 'google_api_key':
    client_ai = genai.Client(api_key=api_key)
  else:
    print('chave não encontrada')

  # Retornar instancia do cliente da IA.
  return client_ai

  # Inicializar o cliente da API.
  client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=groq_api_key
  )
  return client_ai

def markdown_prepare(papel, categoria, contexto, dificuldade, instrucao):
  # Preparação do prompt em Markdown
  prompt_usuario = f"""
  ### Papel:
  {papel}

  ### Área de atuação:
  {categoria}

  ### Contexto:
  {contexto}

  ### Nível de Dificuldade:
  Com valores de 1 a 4, onde 1 indica o mais simples e 4 o mais complexo.
  {dificuldade}

  ### Instrução:
  {instrucao}
  """

  return prompt_usuario

def questions_submit(client_ai, model_id, df_questions):
    # Criar lista vazia para guardar as respostas.
    responses = []

    # Repetição para percorrer todo Dataframe.
    for index, row in df_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      questao = row['question']
      papel = row['system']
      categoria = row['category']
      contexto = row['statement']
      dificuldade = row['level']
      instrucao = row['turns']

      # Preencher prompt do usuário.
      prompt_usuario = markdown_prepare(papel, categoria, contexto, dificuldade, instrucao)

      # Realizar consulta a IA.
      completion = client_ai.chat.completions.create(
          model= model_id,
          messages=[
            # Por questões de sintaxes informo a role, pois é um campo obrigatório,
            # porém o conteúdo é somente o do Markdown.
            {"role": "user", "content": prompt_usuario}
          ],
          max_tokens=1024,
          temperature=0.1 # Conservador.
      )
      response = completion.choices[0].message.content

      # Armazenar o resultado em uma lista.
      responses.append({"question": questao, "response": response})
      print(f"Questão {questao} processada com sucesso.")

    # Fechar conexão com a IA (somente se você a usou ativamente)
    client_ai.close()

    # por motivo de performance as repostas foram adicionadas a uma lista.
    # No retorno a lista é convertida para um dataframe.
    return pd.DataFrame(responses)

# Função para juntar dois Dataframes por um campo chave e opcionalmente selecionar colunas.
def merge_dataframes(df1, df2, key, columns=None):
  merged_df = pd.merge(df1, df2, on=key, how='inner')
  if columns is not None:
    return merged_df[columns]
  return merged_df

Realizar consulta usando o modelo llama 3 de 8 bilhões de parâmetros.

In [ ]:
# Modelo gpt-oss de 20 bilhões de parâmetros.
model_id = 'llama-3.1-8b-instant'

# Dataframe com as respostas do primeiro modelo
df_llama3_response = questions_submit(client_ai_instance('groq_api_key'), model_id, df_my_questions)

Modelo llama 4 scout de 17 bilhões de parâmetros.

In [ ]:
# Modelo llama 4 scout, mais leve, de 17 bilhões de parâmetros.
model_id = 'meta-llama/llama-4-scout-17b-16e-instruct'

# Dataframe com as respostas do primeiro modelo
df_llama4_response = questions_submit(client_ai_instance('groq_api_key'), model_id, df_my_questions)

Específico para o gemini

In [ ]:
# Função para usar modelos de IA específica ao Gemini.
def questions_gemini_submit(client_ai, model_id, df_questions):
  # Criar uma lista vazia, para guardar as respostas, por questão de performance.
  gemini_responses = []

  # Repetição que percorre todo Dataframe.
  for index, row in df_my_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      questao = row['question']
      papel = row['system']
      categoria = row['category']
      contexto = row['statement']
      dificuldade = row['level']
      instrucao = row['turns']

      # Preparação do prompt em Markdown.
      prompt_usuario = markdown_prepare(papel, categoria, contexto, dificuldade, instrucao)

      # Chamada simples para a API da Google em nuvem.
      response = client_ai.models.generate_content(
          model=model_id,
          contents=prompt_usuario,
          config={
              "temperature": 0.1,  # Conservador.
              "max_output_tokens": 1024
          }
      )

      # Armazenar o resultado em uma lista.
      gemini_responses.append({"question": questao, "response": response.candidates[0].content.parts[0].text})
      print(f"Questão {questao} processada com sucesso.")

  # Fechar conexão com a IA
  client_ai.close()

  # Para melhor visualização, converter as respostas em um Dataframe.
  return pd.DataFrame(gemini_responses)

Modelo gemini flash lançado em 2026. Gemini flash é o modelo mais leve quando comparado ao gemini pro.

In [ ]:
# O modelo escolhi do para rodar é o Gemini da Google em nuvém preview de 2026.
model_id = 'gemini-3.1-flash-lite-preview'

# Dataframe com as respostas do primeiro modelo
df_gemini_response = questions_gemini_submit(client_ai_instance('google_api_key'), model_id, df_my_questions)

Arrumar as respostas das 3 IAs.

In [ ]:
# Renomear colunas dos dataframes para melhor legibilidade.
df_llama3_response.rename(columns={'response': 'llama3'}, inplace=True)
df_llama4_response.rename(columns={'response': 'llama4'}, inplace=True)
df_gemini_response.rename(columns={'response': 'gemini'}, inplace=True)
key = 'question'

# Realizar o primeiro inner join entre gptoss_responses e llama4_responses
df_llamas = merge_dataframes(df_llama3_response, df_llama4_response, key)

# Realizar o segundo inner join com df_gemini_response
df_llamas_gemini = merge_dataframes(df_llamas, df_gemini_response, key)

# Junção com o Dataframe de respostas das IAs com o das perguntas, selecionando a coluna choices da linha de guia de respostas.
columns = ['question', 'statement', 'turns', 'level', 'llama3', 'llama4', 'gemini', 'choices']
df_ias = merge_dataframes(df_llamas_gemini, df_my_questions, key, columns)

Métrica usada para avaliar as respostas, BERTScore.

In [ ]:
# Função para calular a métrica, um valor entre 0 e 100%.
def calculate_bert(candidates, references):
  if not isinstance(candidates, list) or not isinstance(references, list):
    print("Candidates e References precisam ser lista.")
    return []
  if len(candidates) != len(references):
    print("Candidates e References precisam ter a mesma quantidade de elementos.")
    return []

  # Conferir se os valores são strings.
  processed_candidates = []
  for item in candidates:
    if isinstance(item, (str, int, float)):
      processed_candidates.append(str(item))
    else:
      processed_candidates.append("")

  processed_references = []
  for item in references:
    if isinstance(item, (str, int, float)):
      processed_references.append(str(item))
    else:
      processed_references.append("")

  # Depois das validações, realizao o calculo propriamente dito.
  try:
    P, R, F1 = score(processed_candidates, processed_references, lang="pt", verbose=False)
    return [f * 100 for f in F1.tolist()]
  except Exception as e:
    print(f"Erro ao tentar calcular BERTScore: {e}")
    return []

Calcular a métrica BERTStore para o Dataframe das respostas das IAs.

In [ ]:
df_ias['llama3_bert'] = pd.Series(calculate_bert(df_ias['llama3'].tolist(), df_ias['choices'].tolist())).round(2)
df_ias['llama4_bert'] = pd.Series(calculate_bert(df_ias['llama4'].tolist(), df_ias['choices'].tolist())).round(2)
df_ias['gemini_bert'] = pd.Series(calculate_bert(df_ias['gemini'].tolist(), df_ias['choices'].tolist())).round(2)

In [ ]:
# Níveis de dificuldade.
difficulty_map = {
    1: 'Estagiário',
    2: 'Analista',
    3: 'Juiz',
    4: 'Ministro'
}

df_ias['level_name'] = df_ias['level'].map(difficulty_map)

Avaliar as respostas por um juiz on-line (IA).

In [ ]:
def llm_judge(client_ai, model_id, name_ia):
  # Criar lista vazia para guardar as respostas.
  responses = []

  # Repetição para percorrer todo Dataframe.
  for index, row in df_ias.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    pergunta = row['statement'] + ' ' + row['turns']
    baseline = row['choices']
    ia = row[name_ia]

    prompt_usuario = f"""
    Você é um jurista sênior especializado em Direito Administrativo e Tributário brasileiro.
    Sua função é atuar como juiz em um benchmark de IAs.
    Analise as respostas comparando-as rigorosamente com a Resposta Base (Gabarito).
    Atribua um percentual de acerto (0-100%). Traga apensa um número com duas casas decimais.
    PERGUNTA: {pergunta}

    RESPOSTA BASE (GABARITO): {baseline}
    ---
    RESPOSTA IA: {ia}
      ---
    [FORMATO DE RESPOSTA]
    Apenas um número com duas casas decimais.
    """
    # Realizar consulta a IA.
    completion = client_ai.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=0.1 # Conservador
    )
    response = completion.choices[0].message.content

    # Armazenar o resultado em uma lista.
    responses.append({"question": questao, "response": response})
    print(f"Questão {questao} processada com sucesso.")

  # Fechar conexão com a IA (somente se você a usou ativamente)
  client_ai.close()

  # por motivo de performance as repostas foram adicionadas a uma lista.
  # No retorno a lista é convertida para um dataframe.
  return pd.DataFrame(responses)

In [ ]:
#model_id = "nvidia/nemotron-3-super-120b-a12b:free"
#df_score_llama3 = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'llama3')
#df_score_llama4 = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'llama4')
#df_score_gemini = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'gemini')

In [ ]:
from IPython.display import HTML
from datetime import date
import matplotlib.pyplot as plt
import io
import base64

# Get today's date
today_date = date.today().strftime('%d/%m/%Y')

# Part 1: Generate the table HTML
table_html_df_display = df_ias[['question', 'level_name', 'llama3_bert', 'llama4_bert', 'gemini_bert']].copy()

# Rename columns for display
table_html_df_display.rename(columns={
    'question': 'Questão',
    'level_name': 'Dificuldade',
    'llama3_bert': 'Llama3 (%)',
    'llama4_bert': 'Llama4 (%)',
    'gemini_bert': 'Gemini (%)'
}, inplace=True)

table_html_string = table_html_df_display.to_html(index=False, classes='', escape=False)

# Part 2: Generate the chart and embed as base64
plt.figure(figsize=(12, 6))
plt.plot(df_ias['question'], df_ias['llama3_bert'], label='Llama3 BERT Score')
plt.plot(df_ias['question'], df_ias['llama4_bert'], label='Llama4 BERT Score')
plt.plot(df_ias['question'], df_ias['gemini_bert'], label='Gemini BERT Score')
plt.title('BERT Score Comparativo entre Modelos de IA')
plt.xlabel('Questão')
plt.ylabel('BERT Score (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(df_ias['question']) # Ensure all question numbers are shown
plt.tight_layout()

buf = io.BytesIO()
plt.savefig(buf, format='png', bbox_inches='tight')
plt.close() # Close the plot to prevent it from displaying directly in the notebook output
image_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')
chart_html_bert = f'<img src="data:image/png;base64,{image_base64}" alt="BERT Score Chart" style="max-width:100%; height:auto;">'

# Part 3: Integrate into the main HTML content
html_report = f"""
<html>
    <head>
        <style>
            body {{ font-family: sans-serif; line-height: 1.6; color: #333; margin: 20px; background-color: #f8f8f8; }}
            .container {{ max-width: 800px; margin: auto; background: #fff; padding: 30px; border-radius: 8px; box-shadow: 0 0 15px rgba(0,0,0,0.1); }}
            h1 {{ color: #0056b3; border-bottom: 2px solid #0056b3; padding-bottom: 10px; margin-bottom: 20px; text-align: center; }}
            h2 {{ color: #0056b3; margin-top: 30px; border-bottom: 1px solid #eee; padding-bottom: 5px; }}
            ul {{ list-style-type: disc; margin-left: 20px; padding-left: 0; }}
            li {{ margin-bottom: 10px; }}
            strong {{ color: #0056b3; }}
            .accuracy-box {{ background-color: #e6f7ff; border: 1px solid #b3e0ff; padding: 15px; border-radius: 5px; text-align: center; margin-top: 20px; font-size: 1.2em; }}
            .accuracy-value {{ font-size: 1.8em; font-weight: bold; color: #007bff; }}
            .plot-container {{ text-align: center; margin-top: 20px; }}
            .plot-container img {{ max-width: 100%; height: auto; border: 1px solid #ddd; border-radius: 5px; padding: 5px; background: #fff; }}
            .info-block {{ background-color: #f0f8ff; border: 1px solid #cceeff; padding: 15px; border-radius: 5px; margin-top: 20px; margin-bottom: 20px; font-size: 0.95em; line-height: 1.5; }}
            .info-block p {{ margin: 5px 0; }}
            /* Table Styling */
            .dataframe {{ border-collapse: collapse; margin-top: 20px; width: 100%;}}
            .dataframe th, .dataframe td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
            .dataframe thead th {{
                background-color: #00008B; /* Dark blue */
                color: white; /* White text for contrast */
            }}
            .dataframe tbody tr {{ background-color: transparent !important; }}
            .dataframe tr:hover {{ background-color: #f1f1f1; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Avaliação de respostas de questões subjetivas de exames passados da OAB por um modelo de IA</h1>

            <div class="info-block">
                <p><strong>Aluno:</strong> Eduardo Henrique</p>
                <p><strong>Data de realização de teste:</strong> {today_date}</p>
                <p><strong>Código fonte:</strong> <a href="https://github.com/eduoududu/juridico" target="_blank">https://github.com/eduoududu/juridico</a></p>
            </div>

            <h2>Etapas Realizadas:</h2>
            <ul>
                <li><strong>Importação do conjunto de dados de perguntas da OAB:</strong> o conjunto de dados de perguntas de provas passadas da OAB foi importado do GitHub, disponível publicamente no endereço web: <a href="https://github.com/maritaca-ai/oab-bench/blob/main/data/judge_prompts.jsonl" target="_blank">https://github.com/maritaca-ai/oab-bench/blob/main/data/judge_prompts.jsonl</a>.</li>
                <li><strong>Importação do dataset de linha guia:</strong> o dataset de linha guia, contendo as respostas relacionadas às perguntas, foi importado do mesmo projeto do GitHub, pela URL: <a href="https://github.com/maritaca-ai/oab-bench/blob/main/data/oab_bench/reference_answer/guidelines.jsonl" target="_blank">https://github.com/maritaca-ai/oab-bench/blob/main/data/oab_bench/reference_answer/guidelines.jsonl</a>.</li>
                <li><strong>Adição de números às questões:</strong> foram adicionados números às questões para facilitar a leitura humana, considerando que o contador em listas no Python inicia do zero.</li>
                <li><strong>Análise e classificação do nível de dificuldade:</strong> foi realizada uma análise manual e interativa, com o auxílio de IA, para classificar o nível de dificuldade das questões, e essa informação foi incluída no conjunto de perguntas.</li>
                <li><strong>Interseção entre dataframes:</strong> foi realizada uma interseção entre os dataframes de questões e de linha guia para reunir todas as informações relevantes em um único dataframe.</li>
                <li><strong>Seleção de subconjunto de questões:</strong> por fim, um subconjunto das questões, especificamente da questão 31 à 40, foi retirado para análises mais focadas.</li>
                <li><strong>Organização em Notebooks Separados:</strong> para otimização e organização, o arquivo responsável pela extração, limpeza e carga dos dados foi separado em um notebook à parte.</li>
                <li><strong>Processamento de Questões por IA:</strong> outro notebook utilizou os dados do primeiro e percorreu todo o conjunto, submetendo as questões a diferentes modelos de IA.</li>
                <li><strong>Estruturação da Pergunta em Markdown:</strong> a pergunta foi cuidadosamente estruturada em formato Markdown para maior detalhamento do conteúdo a ser enviado a uma IA via chave de API.</li>
                <li><strong>Conteúdo do Markdown:</strong> o prompt Markdown incluiu os seguintes elementos: papel, categoria, contexto, dificuldade e instrução.</li>
                <li><strong>Observações sobre Questões Tipo 'Peça':</strong> em análise manual dos dados, foi percebido que, do conjunto de 10 linhas, duas são peças (informação retirada do campo 'statement'). Nesses casos, o campo 'turns' (instrução) estava vazio, o que pode indicar que não se trata de uma questão tradicional. Mesmo assim, foram submetidas às IAs para observar o comportamento.</li>
                <li><strong>Modelos de IA Utilizados:</strong> as questões foram enviadas a três modelos de IA: <b>llama-3.1-8b-instant</b>, <b>meta-llama/llama-4-scout-17b-16e-instruct</b> e <b>gemini-3.1-flash-lite-preview</b>. Esses modelos foram escolhidos por suas características e pela possibilidade de uso gratuito nesta atividade.</li>
                <li><strong>Métrica de Avaliação: BERTScore:</strong> métrica quantitativa utilizada para avaliar a acurácia das respostas de cada IA, comparando-as com a linha guia.</li>
                <li><strong>Tentativa de Avaliação por Juiz Online (IA):</strong> foi feita uma tentativa de utilizar um 'Juiz Online', uma quarta IA robusta, para avaliar as respostas. No entanto, as restrições de uso de IAs gratuitas impossibilitaram essa abordagem.</li>
                <li><strong>Relatório Comparativo Final:</strong> ao final, foi gerado um relatório comparativo utilizando os resultados do BERTScore.</li>
            </ul>
            <h2>Resultados Encontrados:</h2>
            <div class="plot-container">
                <h3>Gráfico de Acurácia Geral</h3>
                {chart_html_bert}
            </div>
            <h2>Pontuação das Respostas namétrica BERTScore</h2>
            {table_html_string}
        </div>
    </body>
</html>
"""
HTML(html_report)

In [ ]:
with open('/content/relatorio_analise_ia.html', 'w') as f:
    f.write(html_report)
print('Report saved to /content/relatorio_analise_ia.html')